# HW3 Problem 2


In [ ]:
!pip install syncode==0.4.16

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 134.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 136.2 MB/s  0:00:00
  Created wheel for syncode: filename=syncode-0.4.16-py3-none-any.whl size=237009 sha256=bcbaa0fa9e4457ba8ea5a4d44820271d223bca9932f5a8c3e4a617d417b7d38d
  Stored in directory: /root/.cache/pip/wheels/8f/1c/98/a275bf3aa3e95ed9b4c5dd487d5ecd3b37a3e64cc39962cd67
Successfully built syncode
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
      Successfully uninstalled regex-2025.11.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstall

In [2]:
import re
from syncode import Syncode

## Grammar


In [3]:
grammar = r"""
start: number op number "\n"?

number: NUM
op: "+" | "-" | "*" | "/"

NUM: "-"? INT ("." INT?)?
INT: DIGIT+
DIGIT: "0".."9"

%ignore " "
"""

In [4]:
model_name = "microsoft/phi-2"

syn_llm = Syncode(model=model_name, grammar=grammar, mode="grammar_mask", max_new_tokens=100)

[2026-10-27 17:48:18,986-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-27 17:49:57,467-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_4841570145_50257.pkl exist: False
[2026-10-27 17:49:57,468-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_4841570145_50257.pkl.
[2026-10-27 17:49:58,064-syncode.mask_store.fsm_set] - 7 FSMs with 19 states initialized in 0.00 seconds
[2026-10-27 17:49:58,065-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-27 17:49:58,073-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 49 to 16


100%|██████████| 19/19 [00:05<00:00,  3.27it/s]

[2026-10-27 17:50:03,929-syncode.mask_store.mask_store] - Time taken to create mask store: 6.45 seconds


In [5]:
prompt = """Extract the two numbers and the operator from each question.

Instruct: What is 327. multiplied by 11.0?
Output: 327. * 11.0

Instruct: What is 45.1 plus 23.54?
Output: 45.1 + 23.54

Instruct: What is 120.4 divided by 4.0?
Output: 120.4 / 4.0

"""

In [6]:
from syncode.larkm import Lark

parser = Lark(grammar, keep_all_tokens=True)

def run_syncode(textual_query):
    full_prompt = prompt + "Instruct: " + textual_query + "\nOutput:"
    output = syn_llm.infer(full_prompt)[0]
    print("Output:", output.strip())

    tree = parser.parse(output.strip())
    num1 = float(tree.children[0].children[0])
    op = str(tree.children[1].children[0])
    num2 = float(tree.children[2].children[0])

    if op == "+":
        answer = num1 + num2
    elif op == "-":
        answer = num1 - num2
    elif op == "*":
        answer = num1 * num2
    else:
        answer = num1 / num2

    return [num1, num2], op, answer

## Test

In [7]:
print(run_syncode("What is 75.5 minus 11.25?"))
print(run_syncode("What is 15.5 minus 60.0?"))
print(run_syncode("What is 3.75 plus 0.5?"))
print(run_syncode("What is 2048.1 multiplied by 31.2?"))
print(run_syncode("What is 81.9 divided by 0.9?"))
print(run_syncode("What is 1000. minus 0.75?"))
print(run_syncode("What is 7. times 9.2?"))
print(run_syncode("What is 14.5 subtracted from 26.0?"))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output: 75.5 - 11.25
([75.5, 11.25], '-', 64.25)
Output: 15.5 - 60.0
([15.5, 60.0], '-', -44.5)
Output: 3.75 + 0.5
([3.75, 0.5], '+', 4.25)
Output: 2048.1 * 31.2
([2048.1, 31.2], '*', 63900.719999999994)
Output: 81.9 / 0.9
([81.9, 0.9], '/', 91.0)
Output: 1000. - 0.75
([1000.0, 0.75], '-', 999.25)
Output: 7. * 9.2
([7.0, 9.2], '*', 64.39999999999999)
Output: 26.0 - 14.5
([26.0, 14.5], '-', 11.5)
